# Multi-Agent LLM Discussion for Paper Reviewing

**Workshop — 30 to 60 minutes**

In this workshop you will build a multi-agent peer-review system using the [SynDisco](https://github.com/dimits-ts/syndisco) library. By the end you will have a working Python script that simulates structured academic peer review through multiple LLM agents — and you will have seen, concretely, why that produces better critiques than a single prompt.

---

## Learning outcomes

By the end of this session you will be able to:

1. **Design** a role-based multi-agent discussion system
2. **Implement** a multi-round interaction loop using SynDisco's Python API
3. **Compare** single-prompt vs multi-agent outputs side by side
4. **Justify** when synthetic discussion is appropriate, especially for structured reasoning tasks and simulations of institutional processes such as peer review

---

## Prerequisites

- Basic Python
- Basic familiarity with LLM / transformer APIs

---

## System design

We will build the following agent structure:

| Role | Responsibility |
|---|---|
| **Methodology Reviewer** | Evaluates experimental design, statistical rigour, reproducibility |
| **Domain Reviewer** | Assesses domain-specific correctness and relevance to the literature |
| **Skeptical Reviewer** | Challenges assumptions and probes for weaknesses |
| **Meta-Reviewer** | Synthesises all reviews into a final structured judgment |

The discussion runs in three rounds:

- **Round 1** — Each reviewer reads the abstract and produces an independent first-pass critique (seed opinions)
- **Round 2** — Each reviewer sees the others' round-1 critiques and updates their position
- **Round 3** — The meta-reviewer reads all prior turns and produces a final recommendation

## 0. Installation

Run the cell below once to install SynDisco and its dependencies.

%pip install syndisco --quiet

In [1]:
%set_env CUDA_VISIBLE_DEVICES=6

env: CUDA_VISIBLE_DEVICES=6


## 1. Configuration

The default abstract is a short excerpt you can replace with any paper of your choice.

In [2]:
import syndisco
from syndisco import TransformersModel


# --- Paper abstract to review -----------------------------------------------
ABSTRACT = """
Title: Attention Is All You Need

Abstract:
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and
convolutions entirely. Experiments on two machine translation tasks show these
models to be superior in quality while being more parallelizable and requiring
significantly less time to train.
""".strip()

model = TransformersModel(
    model_path="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    name="base_model",
    max_out_tokens=1500,
)
syndisco.logging_setup(
    print_to_terminal=True,
    write_to_file=False,
    level="warning",
    use_colors=True
)


## 2. Baseline — single-prompt review

Before building the multi-agent system, ask a single LLM instance to review the paper with one prompt.  
Keep the output in mind — we will compare it with the multi-agent result at the end.

In [3]:
BASELINE_SYSTEM = "You are an expert academic reviewer. Review the paper excerpt provided."
BASELINE_USER   = f"Please review the following paper abstract:\n\n{ABSTRACT}"

baseline_review = model.prompt(BASELINE_SYSTEM, BASELINE_USER)

print("=" * 60)
print("BASELINE — single-prompt review")
print("=" * 60)
print(baseline_review)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BASELINE — single-prompt review
### Review of the Paper Abstract: "Attention Is All You Need"

#### Strengths:
1. **Innovative Approach**: The abstract clearly highlights the novelty of the proposed model, the Transformer, which is based solely on attention mechanisms. This sets it apart from existing models that rely on recurrent or convolutional neural networks.
2. **Simplicity Emphasis**: The phrase "new simple network architecture" emphasizes the simplicity of the proposed model, which is a significant selling point in the field of deep learning.
3. **Performance Claims**: The abstract mentions that the Transformer outperforms existing models in terms of quality, which is a strong claim that would attract readers interested in performance improvements.
4. **Parallelizability and Training Time**: The abstract also points out the benefits of the model in terms of parallelizability and reduced training time, which are important practical considerations for large-scale applications.

#

> **Pause and reflect.** How specific is the critique? Does it identify concrete weaknesses, or does it remain general? Does it consider multiple perspectives?

---

## 4. Defining the reviewer agents

Each `Actor` receives:

- **`persona`** — a dict of attributes that shape its voice (role, expertise, style)
- **`context`** — what the discussion is about (shared across all agents)
- **`instructions`** — what the agent should actually *do* in each turn

The three **reviewer** agents are standard discussion participants (`is_annotator=False`).  
The **meta-reviewer** is also a participant here — it joins round 3 via the turn order we control.

In [4]:
from syndisco import Actor

# Shared context injected into every agent's system prompt
REVIEW_CONTEXT = (
    f"You are participating in a structured academic peer-review discussion. "
    f"The paper under review is:\n\n{ABSTRACT}"
)

# --- Methodology Reviewer ---------------------------------------------------
methodology_reviewer = Actor(
    model=model,
    persona={
        "role": "Methodology Reviewer",
        "expertise": "experimental design, statistics, reproducibility",
        "style": "precise, evidence-focused, constructive",
    },
    context=REVIEW_CONTEXT,
    instructions=(
        "Evaluate the paper's methodology. Focus on experimental design, "
        "evaluation metrics, reproducibility, and statistical rigour. "
        "Be specific: quote or reference parts of the abstract where relevant. "
        "Keep your response to 3-5 sentences."
    ),
    name="MethodologyReviewer",
)

# --- Domain Reviewer --------------------------------------------------------
domain_reviewer = Actor(
    model=model,
    persona={
        "role": "Domain Reviewer",
        "expertise": "natural language processing, deep learning, related literature",
        "style": "scholarly, literature-aware, contextual",
    },
    context=REVIEW_CONTEXT,
    instructions=(
        "Assess the paper's domain-specific contributions. Comment on novelty "
        "relative to prior work, correctness of domain claims, and significance "
        "of the contribution. Keep your response to 3-5 sentences."
    ),
    name="DomainReviewer",
)

# --- Skeptical Reviewer -----------------------------------------------------
skeptical_reviewer = Actor(
    model=model,
    persona={
        "role": "Skeptical Reviewer",
        "expertise": "critical analysis, assumption-testing, identifying gaps",
        "style": "challenging, rigorous, devil's advocate",
    },
    context=REVIEW_CONTEXT,
    instructions=(
        "Challenge the paper's assumptions and probe for weaknesses or missing "
        "evidence. Do not accept claims at face value — ask what could go wrong "
        "or what important context is absent. Keep your response to 3-5 sentences."
    ),
    name="SkepticalReviewer",
)

# --- Meta-Reviewer ----------------------------------------------------------
# Joins only in round 3; SynDisco's turn order will handle this.
meta_reviewer = Actor(
    model=model,
    persona={
        "role": "Meta-Reviewer",
        "expertise": "synthesis, editorial judgment, final recommendations",
        "style": "balanced, decisive, structured",
    },
    context=REVIEW_CONTEXT,
    instructions=(
        "You have read all previous reviewer comments. Synthesise the discussion "
        "into a final structured judgment. Your output must contain:\n"
        "  • Summary of main strengths\n"
        "  • Summary of main weaknesses\n"
        "  • Overall recommendation (Accept / Major Revision / Reject) with a one-sentence rationale."
    ),
    name="MetaReviewer",
)

print("Agents created:")
for agent in [methodology_reviewer, domain_reviewer, skeptical_reviewer, meta_reviewer]:
    print(f"  • {agent.get_actor_name()}")

Agents created:
  • MethodologyReviewer
  • DomainReviewer
  • SkepticalReviewer
  • MetaReviewer


## 5. Round 1 — Independent first-pass reviews

In round 1 each of the three reviewers reads the abstract with no prior discussion context.  
We implement this as **seed opinions**: pre-generated opening statements that are injected into the log before the LLM turns begin.

We call `speak(history=None)` manually here to generate those seed opinions, then pass them into the `Discussion` constructor.  This cleanly separates "generate the seeds" from "run the discussion".

In [5]:
print("Generating round 1 independent reviews...\n")

round1_reviewers = [methodology_reviewer, domain_reviewer, skeptical_reviewer]

# Each reviewer speaks independently — no history yet
seed_opinions   = [reviewer.speak(history=None) for reviewer in round1_reviewers]
seed_usernames  = [reviewer.get_actor_name()    for reviewer in round1_reviewers]

for name, opinion in zip(seed_usernames, seed_opinions):
    print(f"{'─' * 60}")
    print(f"[{name}]")
    print(opinion)

print("─" * 60)
print("Round 1 complete.")

Generating round 1 independent reviews...

────────────────────────────────────────────────────────────
[MethodologyReviewer]
The paper "Attention Is All You Need" presents a novel architecture called the Transformer, which relies solely on attention mechanisms. In terms of experimental design, the authors evaluate their model on two machine translation tasks, demonstrating superior performance compared to existing models that use recurrent or convolutional neural networks. They also highlight the benefits of increased parallelizability and reduced training time. However, the paper does not provide detailed information on the reproducibility of the experiments or the statistical significance of the results, which would enhance the robustness of their claims. For instance, reporting p-values or confidence intervals for the performance metrics could offer greater statistical rigour.
────────────────────────────────────────────────────────────
[DomainReviewer]
The paper "Attention Is All 

## 6. Round 2 and Round 3 — Discussion and meta-review

Now we hand over to SynDisco's `Discussion` class.

- The three reviewers are the participants
- The round-1 outputs become **seed opinions**, so every agent starts with the full prior context
- We run **4 prompted turns**: 3 for the reviewers to update their critiques (round 2), then 1 for the meta-reviewer (round 3)
- A custom turn order puts the meta-reviewer last

### Turn order design

`RoundRobin` cycles through actors in the order they are given.  
We provide `[methodology, domain, skeptical, meta]` and set `conv_len=4`, which gives us exactly one full cycle.

In [6]:
from syndisco import Discussion, RoundRobin

# All four agents are participants; the turn order encodes the round structure
all_agents = [
    methodology_reviewer,
    domain_reviewer,
    skeptical_reviewer,
    meta_reviewer,
]

turn_manager = RoundRobin(actors=all_agents)

discussion = Discussion(
    next_turn_manager=turn_manager,
    users=all_agents,
    # Round-1 reviews are injected as seeds so every agent sees them as context
    seed_opinions=seed_opinions,
    seed_opinion_usernames=seed_usernames,
    # Each agent sees the last 6 messages as context (enough for all seeds + some turns)
    history_context_len=6,
    # 3 updated critiques (round 2) + 1 meta-review (round 3)
    conv_len=4,
)

print("Discussion object created.")
print(f"Seed opinions : {len(seed_opinions)}")
print(f"Prompted turns: {discussion.conv_len}")

Discussion object created.
Seed opinions : 3
Prompted turns: 4


In [7]:
print("Running rounds 2 and 3...\n")
discussion.begin(verbose=True)

Running rounds 2 and 3...



  0%|          | 0/4 [00:00<?, ?it/s]

User MethodologyReviewer posted:
The experimental design in the paper effectively compares the proposed
Transformer model against existing architectures on machine
translation tasks, showcasing its superior performance. However, the
methodology could be strengthened by providing more details on the
reproducibility of the experiments and the statistical significance of
the results. Specifically, reporting p-values or confidence intervals
for the performance metrics would enhance the statistical rigour of
the study. Additionally, including a discussion on the computational
resources required for training both the Transformer and traditional
models would offer a more comprehensive comparison. This would help
readers understand the practical implications of using the Transformer
model in real-world applications. 

User DomainReviewer posted:
The paper "Attention Is All You Need" makes a significant contribution
to the field of natural language processing by introducing the
Transformer mode

## 7. Inspecting and saving the logs

`Discussion.get_logs()` returns a `Logs` object that you can iterate, index, and export to JSON.

In [8]:
logs = discussion.get_logs()

print(f"Total log entries: {len(logs)}\n")
print(f"{'Entry':<6} {'Speaker':<22} {'Text preview'}")
print("-" * 80)
for i, entry in enumerate(logs):
    preview = entry['text'][0].replace('\n', ' ')
    print(f"{i:<6} {entry['name']:<22} {preview}...")

Total log entries: 7

Entry  Speaker                Text preview
--------------------------------------------------------------------------------
0      MethodologyReviewer    T...
1      DomainReviewer         T...
2      SkepticalReviewer      T...
3      MethodologyReviewer    T...
4      DomainReviewer         T...
5      SkepticalReviewer      T...
6      MetaReviewer           *...


## 8. Side-by-side comparison

Let's print the baseline single-prompt review alongside the meta-reviewer's final judgment.

In [10]:
# The meta-reviewer is always the last entry in the log
meta_review_entry = logs[-1]

divider = "=" * 60

print(divider)
print("BASELINE — single-prompt review")
print(divider)
print(baseline_review)

print()
print(divider)
print(f"MULTI-AGENT — {meta_review_entry['name']} final judgment")
print(divider)
print(meta_review_entry['text'])

BASELINE — single-prompt review
### Review of the Paper Abstract: "Attention Is All You Need"

#### Strengths:
1. **Innovative Approach**: The abstract clearly highlights the novelty of the proposed model, the Transformer, which is based solely on attention mechanisms. This sets it apart from existing models that rely on recurrent or convolutional neural networks.
2. **Simplicity Emphasis**: The phrase "new simple network architecture" emphasizes the simplicity of the proposed model, which is a significant selling point in the field of deep learning.
3. **Performance Claims**: The abstract mentions that the Transformer outperforms existing models in terms of quality, which is a strong claim that would attract readers interested in performance improvements.
4. **Parallelizability and Training Time**: The abstract also points out the benefits of the model in terms of parallelizability and reduced training time, which are important practical considerations for large-scale applications.

#

## 9. Discussion and reflection

Consider the following questions with your group:

1. **Specificity** — Does the multi-agent review identify more concrete weaknesses than the single-prompt review? Why might that be?

2. **Perspective coverage** — Which aspects of the paper did each reviewer catch that the others missed? Would a single prompt have covered all of them?

3. **Iteration effect** — Compare the round-2 critiques against the round-1 seeds. Did the reviewers update their positions after reading each other? In what direction?

4. **Institutional simulation** — Real peer review is also a multi-agent, multi-round process. What does this exercise tell us about when synthetic discussion is a good model of a real process?

5. **Failure modes** — When would this system *not* work well? Think about: agent agreement bias ("sycophancy cascades"), the quality ceiling imposed by a shared model, and the absence of genuine domain knowledge.

---

## Summary

In this workshop you:

1. Observed that a **single-prompt** review tends to be general and surface-level
2. Built a **role-specialised multi-agent** system where each agent has a focused mandate
3. Used **seed opinions** to inject round-1 independent reviews as shared context
4. Let SynDisco's **`Discussion`** class manage the turn order and context window across rounds 2 and 3
5. Produced a **structured meta-review** that synthesises all perspectives

The key insight is that **structured reasoning emerges from interaction**. Specialised agents constrained to their own perspective surface different aspects of the same paper, and the meta-reviewer can only produce a grounded synthesis because the prior turns have already done the analytical legwork. This mirrors how real peer review works — and explains why it is hard to replicate with a single prompt.